In [15]:
from pyspark.sql import SparkSession
spark1=SparkSession.builder.appName("practice-1").getOrCreate()

In [16]:
from google.colab import files
files.upload()

Saving stores.csv to stores (1).csv


{'stores (1).csv': b'store_id,store_name,city,state,store_type,manager_name\r\nS101,Metro Mart Hyderabad,Hyderabad,Telangana,Supermarket,Rahul Sharma\r\nS102,Metro Mart Bangalore,Bangalore,Karnataka,Supermarket,Priya Reddy\r\nS103,Metro Mart Mumbai,Mumbai,Maharashtra,Hypermarket,Amit Kumar\r\nS104,Metro Mart Chennai,Chennai,Tamil Nadu,Supermarket,Sneha Patel\r\nS105,Metro Mart Delhi,Delhi,Delhi,Hypermarket,Farhan Ali\r\nS106,Metro Mart Pune,Pune,Maharashtra,Mini Store,Neha Singh\r\nS107,Metro Mart Kochi,Kochi,Kerala,Mini Store,Arjun Verma\r\nS108,Metro Mart Jaipur,Jaipur,Rajasthan,Supermarket,Meera Nair'}

In [17]:
store=spark1.read.csv("stores.csv",header=True,inferSchema=True)

In [18]:
product=spark1.read.csv("product.csv",header=True,inferSchema=True)

In [19]:
inventory=spark1.read.csv("inventory.csv",header=True,inferSchema=True)

In [20]:
sales=spark1.read.csv("sales.csv",header=True,inferSchema=True)

In [21]:
suppliers=spark1.read.option("multiline","True").json("suppliers.json")

In [22]:
store.printSchema()
product.printSchema()
inventory.printSchema()
sales.printSchema()
suppliers.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: int

In [23]:
store.count()

8

In [24]:
product.count()

12

In [25]:
inventory.count()

12

In [26]:
sales.count()

15

In [27]:
suppliers.count()

5

In [28]:
suppliers.write.mode("overwrite").parquet("Bronze/suppliers")

In [29]:
inventory.write.mode("overwrite").parquet("Bronze/inventory")

In [30]:
sales.write.mode("overwrite").parquet("Bronze/sales")

In [31]:
product.write.mode("overwrite").parquet("Bronze/product")

In [32]:
product.filter(product.supplier_id.isNull()).show()

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+



In [33]:
inventory.filter(inventory.stock_quantity.isNull()).show()

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+



In [34]:
sales.filter(sales.sale_amount.isNull()).show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL|      UPI|
+-------+--------+----------+----------+-------------+-----------+---------+



In [35]:
sales.filter(sales.payment_m.isNull()).show()

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1010|    S101|      P104|2026-01-16|            2|      14000|     NULL|
+-------+--------+----------+----------+-------------+-----------+---------+



In [36]:
from pyspark.sql.functions import when
product=product.withColumn("data_quality_status",when(product.supplier_id.isNull(),"Incomplete").otherwise("Complete"))

In [37]:
inventory=inventory.withColumn("data_quality_status",when(inventory.stock_quantity.isNull(),"Incomplete").otherwise("Complete"))

In [38]:
sales=sales.withColumn("data_quality_status",when(sales.sale_amount.isNull(),"Incomplete").otherwise("Complete"))

In [39]:
inventory=inventory.na.fill({"stock_quantity":0})

In [40]:
sales=sales.na.fill({"sale_amount":0})

In [41]:
sales=sales.na.fill({"payment_m":"Not Provided"})

In [42]:
product=product.na.fill({"supplier_id":"Unknown"})

In [43]:
store.write.mode("overwrite").parquet("Silver/store")

In [44]:
product.write.mode("overwrite").parquet("Silver/product")

In [45]:
inventory.write.mode("overwrite").parquet("Silver/inventory")

In [126]:
sales.write.mode("overwrite").parquet("Silver/sales")

In [47]:
from pyspark.sql.functions import col
suppliers=suppliers.select(col("supplier_id"),col("supplier_name"),col("city"),col("contact.phone"),col("contact.email"),col("rating"))

In [48]:
suppliers.select("phone").show()

+----------+
|     phone|
+----------+
|9876500011|
|      NULL|
|9876500013|
|9876500014|
|      NULL|
+----------+



In [49]:
suppliers.select("email").show()

+--------------------+
|               email|
+--------------------+
| techsource@mail.com|
|mobileworld@mail.com|
|                NULL|
|      urban@mail.com|
|                NULL|
+--------------------+



In [50]:
suppliers=suppliers.na.fill({"phone":"Not Provided"},{"email":"Not Provided"})

In [51]:
suppliers.write.mode("overwrite").parquet("Silver/suppliers")

In [52]:
product.join(suppliers,product.supplier_id==suppliers.supplier_id,"inner").show()

+----------+------------+-----------+------------+-----------+----------+-------------------+-----------+--------------------+---------+------------+--------------------+------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|supplier_id|       supplier_name|     city|       phone|               email|rating|
+----------+------------+-----------+------------+-----------+----------+-------------------+-----------+--------------------+---------+------------+--------------------+------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|           Complete|       S201|    TechSource India|Hyderabad|  9876500011| techsource@mail.com|   4.5|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|           Complete|       S202|MobileWorld Distr...|Bangalore|Not Provided|mobileworld@mail.com|   4.2|
|      P109|Refrigerator|Electronics|   Whirlpool|       S203|     38000|           Complete|       S203|     

In [53]:
inventory.join(product,inventory.product_id==product.product_id,"inner").show()

+------------+--------+----------+--------------+-------------+-----------+-------------------+----------+------------+-----------+------------+-----------+----------+-------------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+------------+--------+----------+--------------+-------------+-----------+-------------------+----------+------------+-----------+------------+-----------+----------+-------------------+
|       I1004|    S102|      P101|             8|            5| 2026-01-12|           Complete|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|           Complete|
|       I1001|    S101|      P101|            10|            5| 2026-01-10|           Complete|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|           Complete|
|       I1002|    S101|      P102|            25|           

In [54]:
sales.join(store,sales.store_id==store.store_id,"inner").show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+--------+--------------------+---------+-----------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|store_id|          store_name|     city|      state| store_type|manager_name|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+--------+--------------------+---------+-----------+-----------+------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    S102|Metro M

In [55]:
sales_store_join = sales.join(store, sales.store_id == store.store_id, "inner")
retail_df = sales_store_join.join(product, sales.product_id == product.product_id, "inner")
retail_df = retail_df.join(inventory, (sales.product_id == inventory.product_id) & (sales.store_id == inventory.store_id), "inner")
retail_df = retail_df.join(suppliers, product.supplier_id == suppliers.supplier_id, "inner")
retail_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+--------+--------------------+---------+-----------+-----------+------------+----------+------------+-----------+------------+-----------+----------+-------------------+------------+--------+----------+--------------+-------------+-----------+-------------------+-----------+--------------------+---------+------------+--------------------+------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|store_id|          store_name|     city|      state| store_type|manager_name|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|supplier_id|       supplier_name|     city|       phone|               email|rating|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+-------

In [56]:
product.join(suppliers,product.supplier_id==suppliers.supplier_id,"left").filter(suppliers.supplier_id.isNull()).show()

+----------+------------+-----------+---------+-----------+----------+-------------------+-----------+-------------+----+-----+-----+------+
|product_id|product_name|   category|    brand|supplier_id|unit_price|data_quality_status|supplier_id|supplier_name|city|phone|email|rating|
+----------+------------+-----------+---------+-----------+----------+-------------------+-----------+-------------+----+-----+-----+------+
|      P107|       Watch|    Fashion| Fastrack|       S206|      8000|           Complete|       NULL|         NULL|NULL| NULL| NULL|  NULL|
|      P108|    Backpack|    Fashion|Wildcraft|       S206|      2500|           Complete|       NULL|         NULL|NULL| NULL| NULL|  NULL|
|      P111|  Headphones|Electronics|     Sony|       S999|      3000|           Complete|       NULL|         NULL|NULL| NULL| NULL|  NULL|
|      P112|     T-Shirt|    Fashion|     Puma|    Unknown|      1500|         Incomplete|       NULL|         NULL|NULL| NULL| NULL|  NULL|
+----------+-

In [57]:
inventory.join(product,inventory.product_id==product.product_id,"left").filter(product.product_id.isNull())

DataFrame[inventory_id: string, store_id: string, product_id: string, stock_quantity: int, reorder_level: int, last_update: date, data_quality_status: string, product_id: string, product_name: string, category: string, brand: string, supplier_id: string, unit_price: int, data_quality_status: string]

In [58]:
sales.join(product,sales.product_id==product.product_id,"left").filter(product.product_id.isNull()).show()

+-------+--------+----------+----------+-------------+-----------+---------+-------------------+----------+------------+--------+-----+-----------+----------+-------------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|product_id|product_name|category|brand|supplier_id|unit_price|data_quality_status|
+-------+--------+----------+----------+-------------+-----------+---------+-------------------+----------+------------+--------+-----+-----------+----------+-------------------+
| SA1009|    S108|      P120|2026-01-15|            2|      10000|     Cash|           Complete|      NULL|        NULL|    NULL| NULL|       NULL|      NULL|               NULL|
+-------+--------+----------+----------+-------------+-----------+---------+-------------------+----------+------------+--------+-----+-----------+----------+-------------------+



In [59]:
sales.join(store,sales.store_id==store.store_id,"left").filter(store.store_id.isNull()).show()

+-------+--------+----------+---------+-------------+-----------+---------+-------------------+--------+----------+----+-----+----------+------------+
|sale_id|store_id|product_id|sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|store_id|store_name|city|state|store_type|manager_name|
+-------+--------+----------+---------+-------------+-----------+---------+-------------------+--------+----------+----+-----+----------+------------+
+-------+--------+----------+---------+-------------+-----------+---------+-------------------+--------+----------+----+-----+----------+------------+



In [60]:
inventory=inventory.withColumn("stock_status",when(inventory.stock_quantity<=inventory.reorder_level,"Reorder Required").otherwise("Sufficient stock"))
inventory.show()

+------------+--------+----------+--------------+-------------+-----------+-------------------+----------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|    stock_status|
+------------+--------+----------+--------------+-------------+-----------+-------------------+----------------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|           Complete|Sufficient stock|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|           Complete|Sufficient stock|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|           Complete|Reorder Required|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|           Complete|Sufficient stock|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|           Complete|Sufficient stock|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|           Complete|R

In [61]:
product=product.withColumn("price_category",when(product.unit_price>=50000,"Premium").when(product.unit_price>=10000,"Standard").otherwise("Budget"))
product.show()

+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|price_category|
+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|           Complete|       Premium|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|           Complete|      Standard|
|      P103|  Television|Electronics|          LG|       S203|     45000|           Complete|      Standard|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|           Complete|        Budget|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|           Complete|      Standard|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|           Complete|        Budget|
|      P107|       

In [62]:
sales=sales.withColumn("revenue_category",when(sales.sale_amount>=50000,"High Revenue").when(sales.sale_amount>=15000,"Medium Revenue").otherwise("Low Revenue"))
sales.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     Low Revenue|
| SA1006|    S105|      P108|2026-01-13|            5|      1250

In [63]:
from pyspark.sql.functions import month
sales=sales.withColumn("month",month("sale_date"))
sales.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     Low Revenue|    1|
| SA1006|    S10

In [64]:
from pyspark.sql.functions import year
sales=sales.withColumn("Year",year("sale_date"))
sales.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|Year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     

In [65]:
inventory_product=inventory.join(product,inventory.product_id==product.product_id,"left")
inventory_product=inventory_product.withColumn("inventory_value",inventory_product.stock_quantity*inventory_product.unit_price)
inventory_product_cleaned = inventory_product.drop(product["product_id"])
inventory_product_cleaned = inventory_product_cleaned.drop(product["data_quality_status"])
inventory=inventory_product_cleaned.select(col("inventory_id"),col("store_id"),col("product_id"),col("stock_quantity"),col("reorder_level"),col("last_update"),col("data_quality_status"),col("stock_status"),col("inventory_value"))
inventory.show()

+------------+--------+----------+--------------+-------------+-----------+-------------------+----------------+---------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|    stock_status|inventory_value|
+------------+--------+----------+--------------+-------------+-----------+-------------------+----------------+---------------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|           Complete|Sufficient stock|         650000|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|           Complete|Sufficient stock|         625000|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|           Complete|Reorder Required|          21000|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|           Complete|Sufficient stock|         520000|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|           Complete|Su

In [66]:
suppliers=suppliers.withColumn("supplier_quality",when(suppliers.rating>=4.5,"Excellent").when(suppliers.rating>=3,"Good").otherwise("Average"))
suppliers.show()

+-----------+--------------------+---------+------------+--------------------+------+----------------+
|supplier_id|       supplier_name|     city|       phone|               email|rating|supplier_quality|
+-----------+--------------------+---------+------------+--------------------+------+----------------+
|       S201|    TechSource India|Hyderabad|  9876500011| techsource@mail.com|   4.5|       Excellent|
|       S202|MobileWorld Distr...|Bangalore|Not Provided|mobileworld@mail.com|   4.2|            Good|
|       S203|     HomeTech Supply|   Mumbai|  9876500013|                NULL|   4.4|            Good|
|       S204|  Urban Furniture Co|    Delhi|  9876500014|      urban@mail.com|   4.0|            Good|
|       S205|      Fashion Direct|     Pune|Not Provided|                NULL|   3.8|            Good|
+-----------+--------------------+---------+------------+--------------------+------+----------------+



In [67]:
store.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    1|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    1|
|  Rajasthan|    1|
|  Telangana|    1|
|Maharashtra|    2|
+-----------+-----+



In [68]:
product.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|    Fashion|    4|
|Electronics|    5|
|  Furniture|    3|
+-----------+-----+



In [69]:
product.groupBy("brand").count().show()

+------------+-----+
|       brand|count|
+------------+-----+
|        Nike|    1|
|        Sony|    1|
|Urban Ladder|    1|
|        Puma|    1|
|      Lenovo|    1|
| Featherlite|    1|
|     Samsung|    1|
|      Godrej|    1|
|          LG|    1|
|   Wildcraft|    1|
|    Fastrack|    1|
|   Whirlpool|    1|
+------------+-----+



In [70]:
from pyspark.sql.functions import sum
inventory.groupBy("store_id").agg(sum("stock_quantity")).show()

+--------+-------------------+
|store_id|sum(stock_quantity)|
+--------+-------------------+
|    S105|                 50|
|    S102|                 13|
|    S106|                  0|
|    S104|                  4|
|    S107|                  1|
|    S101|                 38|
|    S108|                 12|
|    S103|                 32|
+--------+-------------------+



In [71]:
inventory_product.groupBy("category").agg(sum("stock_quantity")).show()

+-----------+-------------------+
|   category|sum(stock_quantity)|
+-----------+-------------------+
|    Fashion|                 84|
|       NULL|                 12|
|Electronics|                 48|
|  Furniture|                  6|
+-----------+-------------------+



In [72]:
inventory_product.filter(inventory_product.stock_status=="Reorder Required").select(inventory_product.product_name).show()

+------------+
|product_name|
+------------+
|Office Chair|
| Study Table|
|       Watch|
|Refrigerator|
|        Sofa|
+------------+



In [73]:
sales.agg(sum("sale_amount")).show()

+----------------+
|sum(sale_amount)|
+----------------+
|          373000|
+----------------+



In [74]:
sales.groupBy("store_id").agg(sum("sale_amount")).show()

+--------+----------------+
|store_id|sum(sale_amount)|
+--------+----------------+
|    S105|           20000|
|    S102|           65000|
|    S106|           38000|
|    S104|           24000|
|    S107|           32000|
|    S101|          154000|
|    S108|           10000|
|    S103|           30000|
+--------+----------------+



In [75]:
sales_store=sales.join(store,sales.store_id==store.store_id,"left")
sales_store.groupBy("city").agg(sum("sale_amount")).show()

+---------+----------------+
|     city|sum(sale_amount)|
+---------+----------------+
|Bangalore|           65000|
|    Kochi|           32000|
|  Chennai|           24000|
|   Mumbai|           30000|
|     Pune|           38000|
|    Delhi|           20000|
|Hyderabad|          154000|
|   Jaipur|           10000|
+---------+----------------+



In [76]:
sale_product=sales.join(product,sales.product_id==product.product_id,"left")
sale_product.groupBy("category").agg(sum("sale_amount")).show()

+-----------+----------------+
|   category|sum(sale_amount)|
+-----------+----------------+
|    Fashion|           62000|
|       NULL|           10000|
|Electronics|          243000|
|  Furniture|           58000|
+-----------+----------------+



In [77]:
product_revenue=sale_product.groupBy(sales.product_id).agg(sum("sale_amount").alias("total"))
product_revenue.show()

+----------+------+
|product_id| total|
+----------+------+
|      P110| 32000|
|      P105| 12000|
|      P102| 75000|
|      P106| 18000|
|      P107| 24000|
|      P120| 10000|
|      P103|     0|
|      P109| 38000|
|      P104| 14000|
|      P101|130000|
|      P108| 20000|
+----------+------+



In [78]:
sales.groupBy("payment_m").agg(sum("sale_amount")).show()

+------------+----------------+
|   payment_m|sum(sale_amount)|
+------------+----------------+
|        Card|          133000|
|        Cash|           35500|
|Not Provided|           14000|
|         UPI|          190500|
+------------+----------------+



In [79]:
from pyspark.sql.functions import max
product_revenue.agg(max("total")).show()

+----------+
|max(total)|
+----------+
|    130000|
+----------+



In [80]:
sales_store.agg(max("sale_amount")).show()

+----------------+
|max(sale_amount)|
+----------------+
|           65000|
+----------------+



In [81]:
from pyspark.sql.functions import sum, max
s2=sale_product.groupBy("category").agg(sum("sale_amount").alias("total"))
max_total_revenue = s2.agg(max("total")).collect()[0][0]
s2.filter(s2.total == max_total_revenue).select("category", "total").show()

+-----------+------+
|   category| total|
+-----------+------+
|Electronics|243000|
+-----------+------+



In [82]:
store.createOrReplaceTempView("store")
product.createOrReplaceTempView("product")
inventory.createOrReplaceTempView("inventory")
sales.createOrReplaceTempView("sales")
suppliers.createOrReplaceTempView("suppliers")

In [83]:
result1=spark1.sql("select * from sales")
result1.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|Year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     

In [84]:
result2=spark1.sql("select category,count(product_id) from product group by category")
result2.show()

+-----------+-----------------+
|   category|count(product_id)|
+-----------+-----------------+
|    Fashion|                4|
|Electronics|                5|
|  Furniture|                3|
+-----------+-----------------+



In [85]:
result3=spark1.sql("select store.store_name,sum(sales.sale_amount) from sales join store on sales.store_id=store.store_id group by store_name")
result3.show()

+--------------------+----------------+
|          store_name|sum(sale_amount)|
+--------------------+----------------+
|Metro Mart Bangalore|           65000|
|    Metro Mart Kochi|           32000|
|   Metro Mart Jaipur|           10000|
|   Metro Mart Mumbai|           30000|
|     Metro Mart Pune|           38000|
|Metro Mart Hyderabad|          154000|
|    Metro Mart Delhi|           20000|
|  Metro Mart Chennai|           24000|
+--------------------+----------------+



In [86]:
result5=spark1.sql("select store.city ,sum(sales.sale_amount) from sales join store on sales.store_id=store.store_id group by city")
result5.show()

+---------+----------------+
|     city|sum(sale_amount)|
+---------+----------------+
|Bangalore|           65000|
|    Kochi|           32000|
|  Chennai|           24000|
|   Mumbai|           30000|
|     Pune|           38000|
|    Delhi|           20000|
|Hyderabad|          154000|
|   Jaipur|           10000|
+---------+----------------+



In [87]:
result6=spark1.sql("select product.product_name from product join inventory on product.product_id=inventory.product_id where inventory.stock_status='Reorder Required'")
result6.show()

+------------+
|product_name|
+------------+
|Office Chair|
| Study Table|
|       Watch|
|Refrigerator|
|        Sofa|
+------------+



In [88]:
result7=spark1.sql("select sales.sale_id from product right join sales on product.product_id=sales.product_id where product.product_id is Null ")
result7.show()

+-------+
|sale_id|
+-------+
| SA1009|
+-------+



In [89]:
result9=spark1.sql("select p.product_name from product p left join suppliers s on p.supplier_id=s.supplier_id where s.supplier_id is null")
result9.show()

+------------+
|product_name|
+------------+
|       Watch|
|    Backpack|
|  Headphones|
|     T-Shirt|
+------------+



In [90]:
result10=spark1.sql("select p.product_name,sum(s.sale_amount) as total_revenue from sales s join product p on s.product_id=p.product_id group by p.product_name order by total_revenue desc limit 5")
result10.show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|      Laptop|       130000|
|      Mobile|        75000|
|Refrigerator|        38000|
|        Sofa|        32000|
|       Watch|        24000|
+------------+-------------+



In [91]:
result11=spark1.sql("select payment_m,sum(sale_amount) as total_revenue from sales group by payment_m")
result11.show()

+------------+-------------+
|   payment_m|total_revenue|
+------------+-------------+
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
|         UPI|       190500|
+------------+-------------+



In [92]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, rank, col

product_sales_revenue = sales.join(product, sales.product_id == product.product_id, "inner") \
    .groupBy("product_name").agg(sum("sale_amount").alias("total_revenue"))

window_spec_product = Window.orderBy(col("total_revenue").desc())
ranked_products_by_revenue = product_sales_revenue.withColumn("rank", rank().over(window_spec_product))
ranked_products_by_revenue.show()

+------------+-------------+----+
|product_name|total_revenue|rank|
+------------+-------------+----+
|      Laptop|       130000|   1|
|      Mobile|        75000|   2|
|Refrigerator|        38000|   3|
|        Sofa|        32000|   4|
|       Watch|        24000|   5|
|    Backpack|        20000|   6|
|       Shoes|        18000|   7|
|Office Chair|        14000|   8|
| Study Table|        12000|   9|
|  Television|            0|  10|
+------------+-------------+----+



In [93]:
store_sales_revenue = sales.join(store, sales.store_id == store.store_id, "inner") \
    .groupBy("store_name").agg(sum("sale_amount").alias("total_revenue"))

window_spec_store = Window.orderBy(col("total_revenue").desc())
ranked_stores_by_revenue = store_sales_revenue.withColumn("rank", rank().over(window_spec_store))
ranked_stores_by_revenue.show()

+--------------------+-------------+----+
|          store_name|total_revenue|rank|
+--------------------+-------------+----+
|Metro Mart Hyderabad|       154000|   1|
|Metro Mart Bangalore|        65000|   2|
|     Metro Mart Pune|        38000|   3|
|    Metro Mart Kochi|        32000|   4|
|   Metro Mart Mumbai|        30000|   5|
|  Metro Mart Chennai|        24000|   6|
|    Metro Mart Delhi|        20000|   7|
|   Metro Mart Jaipur|        10000|   8|
+--------------------+-------------+----+



In [94]:
product_category_revenue = sales.join(product, sales.product_id == product.product_id, "inner") \
    .groupBy("category", "product_name").agg(sum("sale_amount").alias("total_revenue"))

window_spec_category = Window.partitionBy("category").orderBy(col("total_revenue").desc())
ranked_products_in_category = product_category_revenue.withColumn("rank", rank().over(window_spec_category))
ranked_products_in_category.show()

+-----------+------------+-------------+----+
|   category|product_name|total_revenue|rank|
+-----------+------------+-------------+----+
|Electronics|      Laptop|       130000|   1|
|Electronics|      Mobile|        75000|   2|
|Electronics|Refrigerator|        38000|   3|
|Electronics|  Television|            0|   4|
|    Fashion|       Watch|        24000|   1|
|    Fashion|    Backpack|        20000|   2|
|    Fashion|       Shoes|        18000|   3|
|  Furniture|        Sofa|        32000|   1|
|  Furniture|Office Chair|        14000|   2|
|  Furniture| Study Table|        12000|   3|
+-----------+------------+-------------+----+



In [95]:
ranked_products_in_category.filter(ranked_products_in_category.rank == 1).show()

+-----------+------------+-------------+----+
|   category|product_name|total_revenue|rank|
+-----------+------------+-------------+----+
|Electronics|      Laptop|       130000|   1|
|    Fashion|       Watch|        24000|   1|
|  Furniture|        Sofa|        32000|   1|
+-----------+------------+-------------+----+



In [96]:
ranked_products_in_category.filter(ranked_products_in_category.rank <= 3).show()

+-----------+------------+-------------+----+
|   category|product_name|total_revenue|rank|
+-----------+------------+-------------+----+
|Electronics|      Laptop|       130000|   1|
|Electronics|      Mobile|        75000|   2|
|Electronics|Refrigerator|        38000|   3|
|    Fashion|       Watch|        24000|   1|
|    Fashion|    Backpack|        20000|   2|
|    Fashion|       Shoes|        18000|   3|
|  Furniture|        Sofa|        32000|   1|
|  Furniture|Office Chair|        14000|   2|
|  Furniture| Study Table|        12000|   3|
+-----------+------------+-------------+----+



In [103]:
store_sales=sales.join(store,sales.store_id==store.store_id,"inner")
store_sales_revenue=store_sales.groupBy(store.store_id, store.state).agg(sum("sale_amount").alias("total_revenue"))
window_spec=Window.partitionBy(col("state")).orderBy(col("total_revenue").desc())
ranked_stores_by_revenue=store_sales_revenue.withColumn("rank",rank().over(window_spec))
ranked_stores_by_revenue.filter(ranked_stores_by_revenue.rank==1).show()

+--------+-----------+-------------+----+
|store_id|      state|total_revenue|rank|
+--------+-----------+-------------+----+
|    S105|      Delhi|        20000|   1|
|    S102|  Karnataka|        65000|   1|
|    S107|     Kerala|        32000|   1|
|    S106|Maharashtra|        38000|   1|
|    S108|  Rajasthan|        10000|   1|
|    S104| Tamil Nadu|        24000|   1|
|    S101|  Telangana|       154000|   1|
+--------+-----------+-------------+----+



In [105]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Define a window specification to calculate running total by sale_date across all sales
my_window_by_date = (Window.orderBy('sale_date')
                     .rowsBetween(Window.unboundedPreceding, Window.currentRow))

# Calculate the running total revenue by sale_date using the 'sales' DataFrame
running_total_revenue = sales.withColumn('running_total_revenue', F.sum('sale_amount').over(my_window_by_date))

# Show the result
running_total_revenue.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+---------------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|Year|running_total_revenue|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+---------------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|                65000|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|               115000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|               180000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete| 

In [111]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

use_lag=sales.withColumn("lag_sale_amount",F.lag("sale_amount").over(Window.orderBy("sale_date")))
use_lag.show()
use_lead_lag=use_lag.withColumn("lead_sale_amount",F.lead("sale_amount",1).over(Window.orderBy("sale_date")))
use_lead_lag.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+---------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|Year|lag_sale_amount|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+---------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|           NULL|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|          65000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|          50000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|         

In [135]:
use_lead_lag.filter(use_lead_lag.lag_sale_amount<use_lead_lag.lead_sale_amount).show()

+-------+--------+----------+----------+-------------+-----------+---------+-------------------+----------------+-----+----+---------------+----------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|revenue_category|month|Year|lag_sale_amount|lead_sale_amount|
+-------+--------+----------+----------+-------------+-----------+---------+-------------------+----------------+-----+----+---------------+----------------+
| SA1006|    S105|      P108|2026-01-13|            5|      12500|      UPI|           Complete|     Low Revenue|    1|2026|           8000|           38000|
| SA1007|    S106|      P109|2026-01-14|            1|      38000|     Card|           Complete|  Medium Revenue|    1|2026|          12500|           32000|
| SA1012|    S103|      P105|2026-01-18|            1|      12000|     Card|           Complete|     Low Revenue|    1|2026|              0|           16000|
| SA1014|    S105|      P108|2026-02-02|            

In [113]:

sales.write.mode("overwrite").partitionBy("Year", "month").parquet("Gold/sales")
print("Gold sales output written successfully, partitioned by Year and month.")

Gold sales output written successfully, partitioned by Year and month.


In [114]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import to_date
incremental_sales_schema = StructType([
    StructField("sale_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("sale_date", StringType(), True), # Read as string initially
    StructField("quantity_sold", IntegerType(), True),
    StructField("sale_amount", IntegerType(), True),
    StructField("payment_m", StringType(), True)
])

incremental_sales_data = [
    ("SA1016", "S101", "P101", "2026-03-01", 1, 65000, "Card"),
    ("SA1017", "S102", "P102", "2026-03-02", 2, 50000, "UPI"),
    ("SA1018", "S103", "P103", "2026-03-03", 1, 45000, "Cash")
]


incremental_sales_df = spark1.createDataFrame(incremental_sales_data, schema=incremental_sales_schema)


incremental_sales_df = incremental_sales_df.withColumn("sale_date", to_date(incremental_sales_df.sale_date, "yyyy-MM-dd"))


incremental_sales_df.write.mode("overwrite").csv("incremental_sales.csv", header=True)
print("Incremental sales file for March 2026 created successfully.")
incremental_sales_df.show()

Incremental sales file for March 2026 created successfully.
+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1016|    S101|      P101|2026-03-01|            1|      65000|     Card|
| SA1017|    S102|      P102|2026-03-02|            2|      50000|      UPI|
| SA1018|    S103|      P103|2026-03-03|            1|      45000|     Cash|
+-------+--------+----------+----------+-------------+-----------+---------+



In [115]:
# Read the incremental sales file
incremental_sales = spark1.read.csv("incremental_sales.csv", header=True, inferSchema=True)

# Ensure 'sale_date' is of DateType
incremental_sales = incremental_sales.withColumn("sale_date", to_date(incremental_sales.sale_date, "yyyy-MM-dd"))

print("Incremental sales file read successfully.")
incremental_sales.show()

Incremental sales file read successfully.
+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1017|    S102|      P102|2026-03-02|            2|      50000|      UPI|
| SA1018|    S103|      P103|2026-03-03|            1|      45000|     Cash|
| SA1016|    S101|      P101|2026-03-01|            1|      65000|     Card|
+-------+--------+----------+----------+-------------+-----------+---------+



In [130]:
import shutil
import os
from pyspark.sql.functions import year, month, when
initial_sales_count = sales.count()
print(f"Initial sales count (before incremental load): {initial_sales_count}")

incremental_sales_appended = incremental_sales.withColumn("Year", year("sale_date")) \
                                              .withColumn("month", month("sale_date")) \
                                              .withColumn("data_quality_status", when(incremental_sales.sale_amount.isNull(), "Incomplete").otherwise("Complete")) \
                                              .withColumn("revenue_category", when(incremental_sales.sale_amount >= 50000, "High Revenue")
                                                                                 .when(incremental_sales.sale_amount >= 15000, "Medium Revenue")
                                                                                 .otherwise("Low Revenue"))


existing_silver_sales = sales


updated_silver_sales = existing_silver_sales.unionByName(incremental_sales_appended)
silver_sales_path = "Silver/sales"
if os.path.exists(silver_sales_path):
    shutil.rmtree(silver_sales_path)
    print(f"Removed existing directory: {silver_sales_path}")
updated_silver_sales.write.mode("overwrite").parquet("Silver/sales")

print("Incremental sales appended to Silver sales Parquet successfully.")
sales = spark1.read.parquet("Silver/sales")
final_sales_count = sales.count()
print(f"Final sales count (after incremental load): {final_sales_count}")

Initial sales count (before incremental load): 15
Removed existing directory: Silver/sales
Incremental sales appended to Silver sales Parquet successfully.
Final sales count (after incremental load): 18


In [123]:
sales.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|Year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|           Complete|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|           Complete|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|           Complete|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|           Complete|     

In [131]:
from pyspark.sql.functions import sum

# Join updated sales with product data
sale_product = sales.join(product, sales.product_id == product.product_id, "left")

# Recalculate product revenue
product_revenue = sale_product.groupBy(sales.product_id).agg(sum("sale_amount").alias("total"))

print("Product revenue recalculated successfully.")
product_revenue.show()

Product revenue recalculated successfully.
+----------+------+
|product_id| total|
+----------+------+
|      P110| 32000|
|      P105| 12000|
|      P102|125000|
|      P106| 18000|
|      P107| 24000|
|      P120| 10000|
|      P103| 45000|
|      P109| 38000|
|      P104| 14000|
|      P101|195000|
|      P108| 20000|
+----------+------+



In [132]:
from pyspark.sql.functions import sum

# Join updated sales with store data
sales_store = sales.join(store, sales.store_id == store.store_id, "left")

# Recalculate store revenue
store_revenue = sales_store.groupBy(store.store_id).agg(sum("sale_amount").alias("total"))

print("Store revenue recalculated successfully.")
store_revenue.show()

Store revenue recalculated successfully.
+--------+------+
|store_id| total|
+--------+------+
|    S105| 20000|
|    S102|115000|
|    S106| 38000|
|    S104| 24000|
|    S107| 32000|
|    S101|219000|
|    S108| 10000|
|    S103| 75000|
+--------+------+



In [133]:

sales.write.mode("overwrite").partitionBy("Year", "month").parquet("Gold/sales")

print("Gold sales output recreated successfully with updated data.")

Gold sales output recreated successfully with updated data.


In [134]:
print(f"Initial sales count: {initial_sales_count}")
print(f"Final sales count (after incremental load): {final_sales_count}")
print(f"Number of new records added: {final_sales_count - initial_sales_count}")

Initial sales count: 15
Final sales count (after incremental load): 18
Number of new records added: 3
